In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
raw_path = Path("../data/raw")
list(raw_path.glob("*"))

In [ ]:
df = pd.read_csv("../data/processed/cleaned_data.csv")

#preliminary inspection of dataset
df.head()
df.shape
df.info()

In [ ]:
#check for null values
df.isnull().sum().sort_values(ascending=False)

print(df["_id"].head())
print(df["timezone"].head())

In [ ]:
#convert to ensure datatype is datetime
df["connectiontime"] = pd.to_datetime(df["connectiontime"])
df["disconnecttime"] = pd.to_datetime(df["disconnecttime"])

#create duration of charge at station
df["duration_hours"] = (
    df["disconnecttime"] - df["connectiontime"]
).dt.total_seconds() / 3600

#splits dataset into different columns for EDA
df["hour"] = df["connectiontime"].dt.hour
df["day_of_week"] = df["connectiontime"].dt.dayofweek
df["month"] = df["connectiontime"].dt.month
df["date"] = df["connectiontime"].dt.date

In [ ]:


print(df.shape)
print(df.isnull().sum().sort_values(ascending=False))
print("Duplicates:", df.duplicated().sum())
print(df[["kwhdelivered"]].describe())

df["connectiontime"] = pd.to_datetime(df["connectiontime"], errors="coerce")
df["disconnecttime"] = pd.to_datetime(df["disconnecttime"], errors="coerce")
df["donechargingtime"] = pd.to_datetime(df["donechargingtime"], errors="coerce")

In [ ]:
df.groupby("hour").size().plot(kind="bar", title="Sessions by Hour")
plt.show()

In [ ]:
df.groupby("day_of_week").size().plot(kind="bar", title="Sessions by Day of Week")
plt.show()

In [ ]:
df.groupby("date")["kwhdelivered"].sum().plot(title="Daily Energy Delivered")
plt.show()

In [ ]:
df["duration_hours"].hist(bins=1)
plt.title("Session Duration")
plt.show()

In [ ]:
daily = (
    df.groupby("date")
      .agg(
          daily_energy=("kwhdelivered", "sum"),
          session_count=("sessionid", "count"),
          avg_duration=("duration_hours", "mean")
      )
      .reset_index()
)

daily["date"] = pd.to_datetime(daily["date"])
daily = daily.sort_values("date")
daily.head()